In [ ]:
from google.colab import drive
drive.mount('/content/drive')
importj os
import pandas as pd
4
# Auto-find the CSV file anywhere in your Drive
file_path = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if 'astram' in f.lower() and f.lower().endswith('.csv'):
            file_path = os.path.join(root, f)
            break
    if file_path:
        break

print("Found file at:", file_path)

df = pd.read_csv(file_path)
print(df.shape)
print(df['priority'].value_counts())

Mounted at /content/drive
Found file at: /content/drive/MyDrive/Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv
(8173, 46)
priority
High    5030
Low     3141
Name: count, dtype: int64


**Install** **LightGBM**

In [ ]:
!pip install lightgbm -q

Clean and prepare the data

In [ ]:
import pandas as pd

df = df.dropna(subset=['priority'])

df['start_datetime'] = pd.to_datetime(df['start_datetime'], errors='coerce')
df['hour'] = df['start_datetime'].dt.hour
df['day_of_week'] = df['start_datetime'].dt.dayofweek
df['month'] = df['start_datetime'].dt.month

df['zone'] = df['zone'].fillna('unknown')
df['veh_type'] = df['veh_type'].fillna('unknown')
df['corridor'] = df['corridor'].fillna('unknown')

df['target'] = df['priority'].map({'High': 1, 'Low': 0})

feature_cols = ['event_type', 'event_cause', 'requires_road_closure', 'veh_type',
                 'zone', 'corridor', 'latitude', 'longitude', 'hour', 'day_of_week', 'month']

categorical_cols = ['event_type', 'event_cause', 'requires_road_closure', 'veh_type', 'zone', 'corridor']

X = df[feature_cols].copy()
y = df['target']

for col in categorical_cols:
    X[col] = X[col].astype('category')

print(X.shape, y.shape)
print(X.head())

(8171, 11) (8171,)
  event_type        event_cause requires_road_closure       veh_type  \
0  unplanned  vehicle_breakdown                 False            lcv   
1  unplanned  vehicle_breakdown                 False  heavy_vehicle   
2  unplanned             others                 False        unknown   
3  unplanned          tree_fall                  True        unknown   
4  unplanned  vehicle_breakdown                 False    private_bus   

             zone      corridor   latitude  longitude  hour  day_of_week  \
0         unknown   Tumkur Road  13.040004  77.518099  17.0          3.0   
1         unknown    ORR East 1  12.921876  77.645158   4.0          1.0   
2  Central Zone 2  Non-corridor  12.955622  77.585708   6.0          5.0   
3         unknown  Non-corridor  13.006147  77.579435  17.0          3.0   
4         unknown  Non-corridor  12.953980  77.585233   4.0          1.0   

   month  
0    3.0  
1    1.0  
2   11.0  
3    3.0  
4    1.0  


Split into train and test sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train size:', X_train.shape, 'Test size:', X_test.shape)

Train size: (6536, 11) Test size: (1635, 11)


Train the LightGBM model

In [ ]:
import lightgbm as lgb

model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

model.fit(
    X_train, y_train,
    categorical_feature=categorical_cols
)

print("Model trained successfully!")

[LightGBM] [Info] Number of positive: 4024, number of negative: 2512
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000535 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 618
[LightGBM] [Info] Number of data points in the train set: 6536, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.615667 -> initscore=0.471197
[LightGBM] [Info] Start training from score 0.471197
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Model trained successfully!


Check performance

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print("F1 Score:", f1_score(y_test, preds))
print()
print(classification_report(y_test, preds))

Accuracy: 1.0
F1 Score: 1.0

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       629
           1       1.00      1.00      1.00      1006

    accuracy                           1.00      1635
   macro avg       1.00      1.00      1.00      1635
weighted avg       1.00      1.00      1.00      1635



 Save the model to Drive


In [ ]:
import joblib

joblib.dump(model, '/content/drive/MyDrive/severity_model.pkl')
joblib.dump(feature_cols, '/content/drive/MyDrive/feature_cols.pkl')
joblib.dump(categorical_cols, '/content/drive/MyDrive/categorical_cols.pkl')

print("Saved to your Google Drive!")

Saved to your Google Drive!


In [ ]:
import pandas as pd
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance)

latitude                 2073
longitude                2059
hour                      777
month                     425
day_of_week               384
corridor                  167
requires_road_closure      83
event_type                 13
event_cause                11
veh_type                    0
zone                        0
dtype: int32
